# 03. 학습 - mT5-base (Seq2Seq)

`google/mt5-base` 기반 Seq2Seq 파인튜닝  
WandB / Notion Training Datasheet 자동 기록

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from dotenv import load_dotenv
load_dotenv('../.env')
print('환경 설정 완료')

---
## 1. 하이퍼파라미터 설정

### 주요 파라미터 가이드

| 파라미터 | 기본값 | 설명 |
|---|---|---|
| `learning_rate` | 5e-4 | mT5 권장 범위: 1e-4 ~ 1e-3 |
| `num_epochs` | 10 | Early stopping 사용 시 넉넉히 설정 |
| `batch_size` | 32 | mT5-base는 경량(300M)이므로 큰 배치 가능 |
| `num_beams` | 4 | 높을수록 품질↑, 속도↓ |
| `max_input_length` | 512 | mT5 권장 최대 길이 |
| `label_smoothing` | 0.1 | 과적합 방지 |

In [ ]:
from src.training.seq2seq_trainer import (
    Seq2SeqModelConfig, Seq2SeqTrainConfig,
    GenerationConfig, Seq2SeqExperimentConfig,
    Seq2SeqHyperParams,
)

# ============================================================
#  하이퍼파라미터 설정  <- 여기를 수정하세요
# ============================================================
model_cfg = Seq2SeqModelConfig(
    model_name        = "google/mt5-base",
    max_input_length  = 512,
    max_output_length = 128,
)

train_cfg = Seq2SeqTrainConfig(
    num_epochs      = 10,
    batch_size      = 32,
    eval_batch_size = 64,
    learning_rate   = 5e-4,
    warmup_ratio    = 0.05,
    weight_decay    = 0.01,
    grad_accum      = 1,
    fp16            = True,
    label_smoothing = 0.1,
    seed            = 42,
)

gen_cfg = GenerationConfig(
    num_beams             = 4,
    length_penalty        = 1.0,
    no_repeat_ngram_size  = 3,
    max_new_tokens        = 128,
)

# ============================================================
#  실험 정보  <- 매 실험마다 수정하세요
# ============================================================
exp_cfg = Seq2SeqExperimentConfig(
    run_name   = "mt5-base-lr5e4-beam4-ep10",
    dataset    = "DialogSum-KO",
    purpose    = "mT5-base 베이스라인 실험",
    tags       = ["mt5", "seq2seq", "multilingual"],
    output_dir = "../outputs/checkpoints",
)

hp = Seq2SeqHyperParams(
    model=model_cfg, train=train_cfg,
    generation=gen_cfg, experiment=exp_cfg,
)
hp.summary()

---
## 2. WandB / Notion 연동

In [ ]:
import yaml
from src.utils.wandb_logger import WandbLogger
from src.utils.notion_logger import NotionLogger

with open('../configs/mt5_config.yaml') as f:
    base_config = yaml.safe_load(f)

base_config['wandb']['name'] = exp_cfg.run_name
base_config['wandb']['tags'] = exp_cfg.tags

wandb_logger  = WandbLogger(base_config)
notion_logger = NotionLogger()

print(f'WandB run: {wandb_logger.run.url}')

---
## 3. 데이터 로드

In [ ]:
train_df = pd.read_csv('../data/raw/train.csv')
dev_df   = pd.read_csv('../data/raw/dev.csv')
test_df  = pd.read_csv('../data/raw/test.csv')

print(f'train: {len(train_df):,}  |  dev: {len(dev_df):,}  |  test: {len(test_df):,}')

---
## 4. 학습 실행

In [ ]:
from src.training.seq2seq_trainer import MT5Trainer

trainer = MT5Trainer(
    hp            = hp,
    notion_logger = notion_logger,
    wandb_logger  = wandb_logger,
)

scores = trainer.run(train_df, dev_df)

print('\n=== 최종 ROUGE 점수 (MeCab) ===')
for k, v in scores.items():
    print(f'  {k}: {v}')

---
## 5. 제출 파일 생성

In [ ]:
from src.inference.submit import SubmissionGenerator

predictions = trainer.predict(test_df['dialogue'].tolist())

gen = SubmissionGenerator(
    sample_submission_path='../data/raw/sample_submission.csv',
    output_dir='../outputs/predictions',
)
submit_path = gen.save(predictions=predictions, filename=f"{exp_cfg.run_name}.csv")
print(f'제출 파일: {submit_path}')

---
## 6. 마무리

In [ ]:
wandb_logger.finish()
print('완료!')
print(f'WandB : {wandb_logger.run.url}')